<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/torch_ntuple_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
if IN_COLAB:
    import os

    if not os.path.exists(os.path.expanduser("~/.ssh/id_rsa")):
        !ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N "" -q
    else:
        print("Key already exists!")
    !ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts 2>/dev/null
    !cat ~/.ssh/id_rsa.pub

Add key here:
https://github.com/settings/keys

In [ ]:
if IN_COLAB:
    !ssh -T git@github.com
    !git config --global user.email "8896197+MarkusThill@users.noreply.github.com"
    !git config --global user.name "Markus Thill"
    import os

    if not os.path.exists("techdays26"):
        !git clone git@github.com:MarkusThill/techdays26.git
    else:
        !git -C techdays26 pull
    !pip install -e techdays26[dev,lab1]

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from techdays26.ntuple_network import NTupleNetwork
from techdays26.td_agent import TDConnect4AgentTorch
from techdays26.torch_board import BoardBatch
from techdays26.ntuples import NTUPLE_BITIDX_LIST


def best_afterstate_values(
    board: "BoardBatch",
    net: "NTupleNetwork",
    *,
    moves_mask: torch.Tensor | None = None,
    randomize: torch.Tensor | None = None,  # [B] bool (epsilon-greedy)
    use_non_losing: bool = True,
    no_grad: bool = True,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Returns:
        chosen_mv:  [B] int64 one-hot move mask (landing square)
        chosen_val: [B] float32 value (reward if terminal else net score)
    """
    dev = board.all_tokens.device
    B = board.all_tokens.shape[0]

    # Which move set to iterate?
    if moves_mask is None:
        moves_mask = (
            board.generate_non_losing_moves()
            if use_non_losing
            else board.generate_legal_moves()
        )
    moves_mask = moves_mask.to(device=dev, dtype=torch.int64)

    yellow_to_move = (board.moves_left.to(torch.int64) & 1) == 0  # [B] bool

    neg_inf = torch.tensor(float("-inf"), device=dev)
    pos_inf = torch.tensor(float("inf"), device=dev)
    best_val = (
        torch
        .where(yellow_to_move, neg_inf, pos_inf)
        .to(torch.float32)
        .expand(B)
        .clone()
    )
    best_mv = torch.zeros((B,), device=dev, dtype=torch.int64)

    # Uniform random move via reservoir sampling over the iterated moves
    rand_mv = torch.zeros((B,), device=dev, dtype=torch.int64)
    rand_val = torch.full((B,), float("nan"), device=dev, dtype=torch.float32)
    seen = torch.zeros(
        (B,), device=dev, dtype=torch.int32
    )  # number of moves seen so far per board

    ctx = torch.no_grad() if no_grad else torch.enable_grad()
    with ctx:
        for mv in board.iter_move_masks(moves_mask):
            active = mv != 0
            if not active.any():
                break

            # Afterstate
            tmp = type(board)(
                all_tokens=board.all_tokens.clone(),
                active_tokens=board.active_tokens.clone(),
                moves_left=board.moves_left.clone(),
            )
            legal = tmp.play_masks(mv)
            active = active & legal  # defensive

            # Terminal reward: +1/-1/0 or NaN if not done
            r = tmp.reward().to(torch.float32)
            v = net(tmp).to(torch.float32)

            # tiebreak noise (optional)
            eps = 1e-4
            v = v + eps * torch.randn_like(v)

            val = torch.where(torch.isnan(r), v, r)  # [B]

            # --- greedy best update ---
            better = (
                torch.where(yellow_to_move, val > best_val, val < best_val) & active
            )
            best_val = torch.where(better, val, best_val)
            best_mv = torch.where(better, mv, best_mv)

            # --- reservoir sampling (uniform random among legal moves) ---
            # increment seen count for boards where this move exists
            seen = seen + active.to(seen.dtype)  # t in {1..}
            # replace current random choice with prob 1/seen
            # (uniform float u in [0,1); replace if u < 1/t)
            t = seen.to(torch.float32)
            replace = active & (torch.rand((B,), device=dev) < (1.0 / t))
            rand_mv = torch.where(replace, mv, rand_mv)
            rand_val = torch.where(replace, val, rand_val)

    if randomize is None:
        return best_mv, best_val

    randomize = randomize.to(device=dev, dtype=torch.bool)
    chosen_mv = torch.where(randomize, rand_mv, best_mv)
    chosen_val = torch.where(randomize, rand_val, best_val)
    return chosen_mv, chosen_val


@torch.no_grad()
def bootstrap_target_from_afterstate(
    after: BoardBatch,
    target_net: NTupleNetwork,
) -> torch.Tensor:
    """
    Returns:
        V_new: [B] float32
        Uses reward for terminal afterstates, else target_net(after).
    """
    r = after.reward().to(torch.float32)  # +1/-1/0 or NaN
    v = target_net(after).to(torch.float32)  # [-1,1]
    return torch.where(torch.isnan(r), v, r)

In [ ]:
from techdays26.ntuples import NTUPLE_BITIDX_LIST, format_ntuple, ntuple_summary

info = ntuple_summary(NTUPLE_BITIDX_LIST)
print(
    f"N-tuples: {info['count']} x {info['length']}-tuples  (LUT size: {info.get('lut_size')})"
)
print(f"Hash: {info['hash'][:16]}...")
print()

for i, tup in enumerate(NTUPLE_BITIDX_LIST):
    print(f"--- Tuple {i:2d}: {tup} ---")
    print(format_ntuple(tup))
    print()

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent

bitbully_agent_max_depth = BitBully(opening_book="12-ply-dist", tie_break="random")

# Some weaker agents with limited depth
bitbully_agents = {}
for search_depth in (1, 2, 4, 8):
    agent = BitBully(opening_book=None, tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}ply"] = agent

for search_depth in (8, 10):
    agent = BitBully(opening_book="8-ply", tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}-ply-book8ply"] = agent

bitbully_agents[f"bitbully-16ply-book12ply"] = BitBully(
    opening_book="12-ply-dist", tie_break="random", max_depth=16
)
bitbully_agents[f"bitbully-full-strength"] = bitbully_agent_max_depth

In [ ]:
import logging
from techdays26 import bitbully_arena as ba

from techdays26.bitbully_arena import (
    get_table_legend,
    format_aggregate_table,
)


def evaluate(td_agent, rnd_agent, bitbully_agents):
    # Logger is optional; arena is silent by default unless you configure logging.
    logger = logging.getLogger("bitbully.arena")
    logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout

    agent_specs = [
        ba.AgentSpec(
            agent_id=k,
            agent=v,
            colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
            epsilons=(0.00, 0.1, 0.2, 0.3) if "full-strength" in k else (0.00,),
        )
        for k, v in bitbully_agents.items()
    ]
    matchups = (
        [ba.Matchup(yellow_id=k, red_id="ntuple") for k in bitbully_agents.keys()]
        + [ba.Matchup(yellow_id="ntuple", red_id=k) for k in bitbully_agents.keys()]
        + [
            ba.Matchup(yellow_id="ntuple", red_id="random"),
            ba.Matchup(yellow_id="random", red_id="ntuple"),
        ]
    )

    cfg = ba.ArenaConfig(
        agents=(
            *agent_specs,
            ba.AgentSpec(
                agent_id="random",
                agent=rnd_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
                epsilons=(0.00,),
            ),
            ba.AgentSpec(
                agent_id="ntuple",
                agent=td_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both,
                epsilons=(0.00,),
            ),
        ),
        n_games=50,  # n games per pairing per ε per color-swap
        time_control=ba.TimeControl(
            per_move_timeout_s=4.0,  # best-effort (measured after return)
            per_game_budget_s=45.0,  # seconds of total think time
        ),
        matchups=matchups,
        seed=None,
        log_scores=False,  # set True to also call score_all_moves() for logging
        use_tqdm=True,  # optional progress bar
        logger=logger,
    )

    # -----------------------------
    # Run arena
    # -----------------------------

    arena = ba.BitBullyArena()
    result = arena.run(cfg)
    return result

In [ ]:
from techdays26 import utils

commit_sha = utils.get_commit_hash("/content/techdays26" if IN_COLAB else ".")
reqs = utils.get_requirements_string()

In [ ]:
import copy
import json
import platform
import sys
import time
from datetime import datetime
from pathlib import Path

from techdays26 import __version__ as techdays26_version
from techdays26.ntuples import NTUPLE_BITIDX_LIST, ntuple_summary
from techdays26.torch_board import BoardBatch

timestamp = datetime.now().strftime("%Y%m%d_%H-%M")

device = "cuda" if IN_COLAB else "cpu"
pre_trained_model: str | None = None
train_folder: str | Path = (
    Path("/content/drive/MyDrive/models/" if IN_COLAB else f"./") / f"exp_{timestamp}"
)

n_evaluate = 1000
n_steps = 100000
lr_initial = 3e-4
lr_final = 1e-6
gamma = 0.99999  # decay factor for learning rate; use 0.9999 for faster decay in 100k games.
B = 10000
epsilon = 0.1
use_target_net = True  # True = Polyak-averaged (EMA) target network, False = bootstrap from online net
use_online_net_for_action = True  # only relevant if use_target_net=True; if False, use target net for action selection as well (more stable but slower learning)
tau = 0.005  # Polyak averaging coefficient (only used when use_target_net=True). We might want to start with 0.001 first!

train_folder = Path(train_folder)
train_folder.mkdir(parents=False, exist_ok=False)

# load pre-trained, if desired:
if pre_trained_model:
    ntuple_net = NTupleNetwork.load(
        pre_trained_model,
        device=device,
    )
    # ntuple_net.eval() # DO NOT DO DURING TRAINING
else:
    ntuple_net = NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST).to(device)

# Target network: a frozen copy used for computing V_new (bootstrap targets)
if use_target_net:
    target_net = copy.deepcopy(ntuple_net)
    target_net.eval()
    for p in target_net.parameters():
        p.requires_grad_(False)
else:
    target_net = ntuple_net  # same object = plain bootstrapping

torch_board = BoardBatch.empty(B, device)

assert ntuple_net.training, "Model should be in training mode!"

opt = torch.optim.Adam(ntuple_net.parameters(), lr=lr_initial)  # used to be 1e-3
scheduler = torch.optim.lr_scheduler.LambdaLR(
    opt,
    lr_lambda=lambda step: (
        (lr_final / lr_initial) + (1 - lr_final / lr_initial) * pow(gamma, step)
    ),
)

ntuple_info = ntuple_summary(NTUPLE_BITIDX_LIST)

# Save n-tuples to JSON for full reproducibility
ntuples_json_path = train_folder / "0_ntuples.json"
with ntuples_json_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "ntuples": NTUPLE_BITIDX_LIST,
            "summary": {k: v for k, v in ntuple_info.items()},
        },
        f,
        indent=2,
    )

# Save training parameters to JSON for comparison across runs
params_json_path = train_folder / "0_params.json"
with params_json_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "start_time": timestamp,
            "device": device,
            "pre_trained_model": pre_trained_model,
            "n_steps": n_steps,
            "n_evaluate": n_evaluate,
            "batch_size": B,
            "lr_initial": lr_initial,
            "lr_final": lr_final,
            "gamma": gamma,
            "epsilon": epsilon,
            "use_target_net": use_target_net,
            "use_online_net_for_action": use_online_net_for_action
            if use_target_net
            else None,
            "tau": tau if use_target_net else None,
            "optimizer": type(opt).__name__,
            "optimizer_betas": list(opt.defaults["betas"]),
            "optimizer_eps": opt.defaults["eps"],
            "optimizer_weight_decay": opt.defaults["weight_decay"],
            "loss": "MSE",
            "use_non_losing": False,
            "activation": "tanh",
            "mirror_symmetry": True,
            "n_tuples": ntuple_info["count"],
            "tuple_length": ntuple_info["length"],
            "lut_size": ntuple_info.get("lut_size"),
            "total_params": sum(p.numel() for p in ntuple_net.parameters()),
            "ntuple_hash": ntuple_info["hash"],
            "commit_sha": commit_sha,
            "techdays26_version": techdays26_version,
            "python_version": sys.version,
            "pytorch_version": torch.__version__,
        },
        f,
        indent=2,
    )

out_file = Path(train_folder / "0_log.txt")
with out_file.open("a", encoding="utf-8") as f:
    f.write(f"techdays26 version: {techdays26_version}\n")
    f.write(f"Base model: {pre_trained_model}\n")
    f.write(f"Git commit SHA: {commit_sha}\n")
    f.write(f"Requirements: {reqs}\n")
    f.write("\n--- Environment ---\n")
    f.write(f"start_time: {timestamp}\n")
    f.write(f"hostname: {platform.node()}\n")
    f.write(f"python: {sys.version}\n")
    f.write(f"pytorch: {torch.__version__}\n")
    f.write(f"cuda: {torch.version.cuda}\n")
    f.write(f"device: {device}\n")
    f.write("\n--- Training Settings ---\n")
    f.write(f"device: {device}\n")
    f.write(f"batch_size (B): {B}\n")
    f.write(f"n_steps: {n_steps}\n")
    f.write(f"n_evaluate: {n_evaluate}\n")
    f.write(f"lr_initial: {lr_initial}\n")
    f.write(f"lr_final: {lr_final}\n")
    f.write(f"gamma (lr decay): {gamma}\n")
    f.write(f"epsilon (exploration): {epsilon}\n")
    f.write(f"use_target_net: {use_target_net}\n")
    f.write(
        f"use_online_net_for_action: {use_online_net_for_action if use_target_net else 'N/A'}\n"
    )
    f.write(f"tau (Polyak): {tau if use_target_net else 'N/A'}\n")
    f.write(
        f"optimizer: {type(opt).__name__} (betas={opt.defaults['betas']}, eps={opt.defaults['eps']}, weight_decay={opt.defaults['weight_decay']})\n"
    )
    f.write(
        f"scheduler: LambdaLR (lr_final/lr_initial + (1 - lr_final/lr_initial) * gamma^step)\n"
    )
    f.write(f"loss: MSE\n")
    f.write(f"use_non_losing: False\n")
    f.write(f"activation: tanh\n")
    f.write(f"\n--- N-Tuple Info ---\n")
    f.write(f"n_tuples (M): {ntuple_info['count']}\n")
    f.write(f"tuple_length (N): {ntuple_info['length']}\n")
    f.write(f"uniform_length: {ntuple_info['uniform']}\n")
    f.write(f"LUT_size (K=4^N): {ntuple_info.get('lut_size', 'N/A')}\n")
    f.write(f"ntuple_hash: {ntuple_info['hash']}\n")
    f.write(f"total_params: {sum(p.numel() for p in ntuple_net.parameters())}\n")
    f.write(f"mirror_symmetry: True\n")
    f.write("--- End Training Settings ---\n\n")
    f.write(get_table_legend() + "\n\n")

In [ ]:
import json

metrics_json_path = train_folder / "0_metrics.json"
arena_json_path = train_folder / "0_arena_metrics.json"

# Load existing metrics if resuming, otherwise start fresh
if metrics_json_path.exists():
    with metrics_json_path.open("r", encoding="utf-8") as f:
        all_metrics = json.load(f)
else:
    all_metrics = []

if arena_json_path.exists():
    with arena_json_path.open("r", encoding="utf-8") as f:
        all_arena_metrics = json.load(f)
else:
    all_arena_metrics = []


def _fmt_elapsed(seconds: float) -> str:
    """Format seconds as HH:MM:SS."""
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def _arena_result_to_rows(result) -> list[dict]:
    """Extract aggregate rows from an ArenaResult (same fields as format_aggregate_table)."""
    rows = []
    for r in result.aggregates:
        score = int(r.yellow_wins) - int(r.red_wins)
        games = int(r.games)
        rows.append({
            "agent_yellow": r.agent_yellow,
            "agent_red": r.agent_red,
            "epsilon_yellow": float(r.epsilon_yellow),
            "epsilon_red": float(r.epsilon_red),
            "games": games,
            "yellow_wins": int(r.yellow_wins),
            "red_wins": int(r.red_wins),
            "draws": int(r.draws),
            "score": score,
            "avg": (score / games) if games else 0.0,
            "timeouts": int(getattr(r, "timeouts", 0)),
            "illegal_moves": int(getattr(r, "illegal_moves", 0)),
            "exceptions": int(getattr(r, "exceptions", 0)),
            "total_time_s": float(getattr(r, "total_time_s", 0.0)),
        })
    return rows


training_start_time = time.time()
last_eval_time = training_start_time
for step in range(n_steps + 1):
    V_old = ntuple_net(torch_board)
    randomize = (
        torch.rand((B,), device=torch_board.all_tokens.device) < epsilon
    )  # [B] bool

    # 1) choose action with ONLINE net if desired (we will use the target net for bootstrapping the value later, if use_target_net=True)
    with torch.no_grad():
        best_mv, V_new = best_afterstate_values(
            torch_board,
            ntuple_net
            if use_online_net_for_action
            else target_net,  # target_net == ntuple_net when use_target_net=False
            moves_mask=None,
            randomize=randomize,
            use_non_losing=False,
            no_grad=True,
        )

    # At any time, any move has to be legal:
    legal = torch_board.play_masks(best_mv)
    assert legal.all(), "Chosen move must be legal!"
    done = torch_board.done()

    if use_online_net_for_action and use_target_net:
        # 2) compute bootstrap target with TARGET net
        with torch.no_grad():
            V_new = bootstrap_target_from_afterstate(
                after=torch_board,
                target_net=target_net,
            )

    # Update only for afterstates which are terminal states or not random moves
    # TODO: Maybe we can always update. Should not make a difference for epsilon=0.1
    # TODO: Try in an A/B test to always update vs only update non-random moves and terminal states
    update_mask = done | (~randomize)

    loss = torch.nn.functional.mse_loss(V_old[update_mask], V_new[update_mask])

    opt.zero_grad(set_to_none=True)
    loss.backward()

    # Snapshot W before optimizer step to compute ||ΔW|| / ||W||
    if step % (n_evaluate // 10) == 0:
        with torch.no_grad():
            W_mask = ntuple_net.W != 0.0
            W_before = ntuple_net.W[W_mask]  # only consider non-zero weights for norm
            W_before = W_before.data.clone()
            W_norm = W_before.norm().item()

    opt.step()
    scheduler.step()

    # Polyak update: W_target = (1 - tau) * W_target + tau * W
    if use_target_net:
        with torch.no_grad():
            for p_target, p_online in zip(
                target_net.parameters(), ntuple_net.parameters()
            ):
                p_target.data.mul_(1.0 - tau).add_(p_online.data, alpha=tau)

    if step % (n_evaluate // 10) == 0:
        now = time.time()
        elapsed_s = now - training_start_time
        elapsed_fmt = _fmt_elapsed(elapsed_s)
        with torch.no_grad():
            # Compute relative weight update
            W_after = ntuple_net.W[W_mask]
            delta_W_norm = (W_after.data - W_before).norm().item()
            rel_update = delta_W_norm / W_norm if W_norm > 0 else float("inf")

        lr = opt.param_groups[0]["lr"]
        grad = ntuple_net.W.grad
        grad_nz = grad[grad != 0.0]
        upd = update_mask.float().mean().item()
        term = done.float().mean().item()
        n_wins = torch_board.has_win().sum().item()
        rand = randomize.float().mean().item()
        mv_left = torch_board.moves_left

        print(
            f"step {step:6d} | {elapsed_fmt} | lr = {lr:.3e} | loss = {loss.item():.6f} | ||ΔW||/||W|| = {rel_update:.6e} | V_old (min/max): {V_old.min().item():.3f}/{V_old.max().item():.3f} | V_old (mean±std): {V_old.mean().item():.3f}±{V_old.std().item():.3f} | Avg. games done: {done.float().mean(): .2f}"
        )
        print(
            f"Moves left: min/max: {mv_left.min().item()}/{mv_left.max().item()}. mean ± std: {mv_left.float().mean().item(): .2f}±{mv_left.float().std().item():.2f}.\n"
        )

        # Collect metrics snapshot
        m = {
            "step": step,
            "training_elapsed_s": elapsed_s,
            "training_elapsed": elapsed_fmt,
            "wall_time_s": now - last_eval_time,
            "lr": lr,
            "loss": loss.item(),
            "rel_weight_update": rel_update,
            "delta_W_norm": delta_W_norm,
            "W_norm": W_norm,
            "V_old_min": V_old.min().item(),
            "V_old_max": V_old.max().item(),
            "V_old_mean": V_old.mean().item(),
            "V_old_std": V_old.std().item(),
            "V_old_abs_mean": V_old.abs().mean().item(),
            "V_old_abs_std": V_old.abs().std().item(),
            "grad_nnz": int(grad_nz.shape[0]),
            "grad_min": grad_nz.min().item() if grad_nz.numel() > 0 else 0.0,
            "grad_max": grad_nz.max().item() if grad_nz.numel() > 0 else 0.0,
            "grad_mean": grad_nz.mean().item() if grad_nz.numel() > 0 else 0.0,
            "grad_std": grad_nz.std().item() if grad_nz.numel() > 1 else 0.0,
            "update_frac": upd,
            "done_frac": term,
            "randomize_frac": rand,
            "n_wins": n_wins,
            "moves_left_min": int(mv_left.min().item()),
            "moves_left_max": int(mv_left.max().item()),
            "moves_left_mean": mv_left.float().mean().item(),
            "moves_left_std": mv_left.float().std().item(),
        }
        all_metrics.append(m)

        # Write full list atomically
        tmp_path = metrics_json_path.with_suffix(".tmp")
        with tmp_path.open("w", encoding="utf-8") as f:
            json.dump(all_metrics, f, indent=2)
        tmp_path.rename(metrics_json_path)

        # Write human-readable log from the metrics dict
        sep = "=" * 90
        with out_file.open("a", encoding="utf-8") as f:
            f.write(f"{sep}\n")
            f.write(
                f"Step {m['step']:6d}  |  {m['training_elapsed']}  |  dt: {m['wall_time_s']:.1f}s  |  lr = {m['lr']:.3e}  |  loss = {m['loss']:.5f}\n"
            )
            f.write(f"\n")
            f.write(
                f"  V_old        min/max: {m['V_old_min']:.3f} / {m['V_old_max']:.3f}    mean +/- std: {m['V_old_mean']:.3f} +/- {m['V_old_std']:.3f}    |V|: {m['V_old_abs_mean']:.3f} +/- {m['V_old_abs_std']:.3f}\n"
            )
            f.write(
                f"  Gradient     nnz: {m['grad_nnz']}    min/max: {m['grad_min']:.3f} / {m['grad_max']:.3f}    mean +/- std: {m['grad_mean']:.3f} +/- {m['grad_std']:.3f}\n"
            )
            f.write(
                f"  Weights      ||W|| = {m['W_norm']:.6e}    ||dW|| = {m['delta_W_norm']:.6e}    ||dW||/||W|| = {m['rel_weight_update']:.6e}\n"
            )
            f.write(
                f"  Batch        update: {m['update_frac']:.2f}    done: {m['done_frac']:.2f}    random: {m['randomize_frac']:.2f}    wins: {m['n_wins']}\n"
            )
            f.write(
                f"  Moves left   min/max: {m['moves_left_min']} / {m['moves_left_max']}    mean +/- std: {m['moves_left_mean']:.2f} +/- {m['moves_left_std']:.2f}\n"
            )
            f.write(f"{sep}\n\n")

    if step % n_evaluate == 0:
        print("evaluate...")
        # Save the weights to a file
        weights_path = f"{train_folder}/step_{step}_model_weights.pt"
        ntuple_net.save(weights_path)
        td_agent = (
            TDConnect4AgentTorch(  # TODO: Allow to pass object (or copy) directly
                model_path=weights_path
            )
        )
        result = evaluate(td_agent, ba.RandomAgent(), bitbully_agents)
        result.save_json(f"{train_folder}/step_{step}_arena_result.json")

        # Append compact arena scores to cumulative JSON
        all_arena_metrics.append({
            "step": step,
            "training_elapsed_s": time.time() - training_start_time,
            "aggregates": _arena_result_to_rows(result),
        })
        tmp_path = arena_json_path.with_suffix(".tmp")
        with tmp_path.open("w", encoding="utf-8") as f:
            json.dump(all_arena_metrics, f, indent=2)
        tmp_path.rename(arena_json_path)

        with out_file.open("a", encoding="utf-8") as f:
            f.write("=====================================================" * 3 + "\n")
            f.write(format_aggregate_table(result))
            f.write(
                "=====================================================" * 3 + "\n\n"
            )

        last_eval_time = time.time()

    if done.any():
        torch_board.reset(done)

In [ ]:
td_agent = TDConnect4AgentTorch(
    model_path="/home/mthill/techdays26/train_run/step_6000_model_weights_loss_0.016.pt"
)

agents: dict[str, Connect4Agent] = {
    "BitBully-max-depth": bitbully_agent_max_depth,
    "TD-Agent": td_agent,
    "BitBully-4-ply": bitbully_agents["bitbully-8ply"],
}

In [ ]:
from bitbully.gui_c4 import GuiC4

%matplotlib ipympl

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())

In [ ]:
result = evaluate(td_agent, RandomAgent(), bitbully_agent_max_depth)

In [ ]:
print(format_aggregate_table(result))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_adam_effective_lr(optimizer, title_suffix=""):
    """Plot histogram of Adam's per-parameter effective learning rates.

    For Adam, the actual update is:  W -= lr * m_hat / (sqrt(v_hat) + eps)
    So the effective learning rate per weight is:  lr / (sqrt(v_hat) + eps)
    where v_hat = v / (1 - beta2^t)  is the bias-corrected second moment.
    """
    for group_idx, group in enumerate(optimizer.param_groups):
        lr = group["lr"]
        beta1, beta2 = group["betas"]
        eps = group["eps"]

        for p_idx, p in enumerate(group["params"]):
            state = optimizer.state[p]
            if not state:
                print(f"No optimizer state yet (run at least 1 step first)")
                return

            step = state["step"]
            # exp_avg_sq is the raw (non-bias-corrected) second moment
            v = state["exp_avg_sq"]

            # Bias correction
            bias_correction2 = 1.0 - beta2**step

            # Effective LR per weight: lr / (sqrt(v / bias_correction2) + eps)
            v_hat = v / bias_correction2
            eff_lr = lr / (torch.sqrt(v_hat) + eps)

            # Only look at weights that have been touched (v > 0)
            mask = v > 0
            eff_lr_active = eff_lr[mask].detach().cpu().numpy()

            if eff_lr_active.size == 0:
                print("No active weights found.")
                return

            fig, axes = plt.subplots(1, 3, figsize=(16, 4))

            # 1) Histogram of effective LR
            ax = axes[0]
            ax.hist(eff_lr_active, bins=100, edgecolor="black", alpha=0.7)
            ax.set_xlabel("Effective LR")
            ax.set_ylabel("Count")
            ax.set_title(f"Effective LR distribution (step {step})")
            ax.axvline(lr, color="r", linestyle="--", label=f"scheduled lr={lr:.2e}")
            ax.legend()

            # 2) Log-scale histogram
            ax = axes[1]
            log_eff = np.log10(eff_lr_active)
            ax.hist(log_eff, bins=100, edgecolor="black", alpha=0.7, color="orange")
            ax.set_xlabel("log10(Effective LR)")
            ax.set_ylabel("Count")
            ax.set_title(f"log10(Effective LR) distribution")
            ax.axvline(
                np.log10(lr),
                color="r",
                linestyle="--",
                label=f"log10(scheduled lr)={np.log10(lr):.2f}",
            )
            ax.legend()

            # 3) Ratio: effective_lr / scheduled_lr
            ax = axes[2]
            ratio = eff_lr_active / lr
            ax.hist(ratio, bins=100, edgecolor="black", alpha=0.7, color="green")
            ax.set_xlabel("Effective LR / Scheduled LR")
            ax.set_ylabel("Count")
            ax.set_title(f"Ratio effective/scheduled LR")
            ax.axvline(1.0, color="r", linestyle="--", label="ratio = 1")
            ax.legend()

            fig.suptitle(
                f"Adam effective LR analysis{title_suffix}\n"
                f"step={step}, scheduled_lr={lr:.2e}, "
                f"active weights={mask.sum().item()}/{v.numel()}, "
                f"eff_lr: median={np.median(eff_lr_active):.2e}, "
                f"min={eff_lr_active.min():.2e}, max={eff_lr_active.max():.2e}",
                fontsize=10,
            )
            plt.tight_layout()
            plt.show()

            # Also print summary stats of v_hat (the adapted squared gradient)
            v_hat_active = v_hat[mask].detach().cpu().numpy()
            print(
                f"v_hat (bias-corrected 2nd moment) stats for {mask.sum().item()} active weights:"
            )
            print(
                f"  min={v_hat_active.min():.2e}, max={v_hat_active.max():.2e}, "
                f"median={np.median(v_hat_active):.2e}, mean={v_hat_active.mean():.2e}"
            )


plot_adam_effective_lr(opt)

In [ ]:
# =============================================================================
# Tests for NTupleNetwork & best_afterstate_values
# =============================================================================
# Requires: cell-6 (NTUPLE_BITIDX_LIST), cell-7 (NTupleNetwork, best_afterstate_values),
#           cell-9 (bbc import)
# Run these BEFORE training (or restart kernel + run cells 1-9 + this cell).

import os
import tempfile

import bitbully.bitbully_core as bbc
import numpy as np
import torch

from techdays26.ntuples import NTUPLE_BITIDX_LIST
from techdays26.torch_board import BoardBatch, move_mask_to_column

DEVICE = "cpu"


def _mk_net(*, random_weights: bool = False) -> NTupleNetwork:
    """Create a fresh NTupleNetwork, optionally with small random weights."""
    net = NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST)
    if random_weights:
        with torch.no_grad():
            net.W.normal_(0, 0.05)
    net.eval()
    return net


def _build_midgame_boards(B: int, *, n_rounds: int = 15, seed: int = 42) -> BoardBatch:
    """Build a batch of diverse mid-game positions by playing random moves."""
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    board = BoardBatch.empty(B, DEVICE)
    for _ in range(n_rounds):
        actions = torch.randint(0, 7, (B,), device=DEVICE, generator=g)
        board.play_columns(actions)
        board.reset(board.done())
    return board


def _build_synced_boards(
    B: int, *, n_rounds: int = 15, seed: int = 42
) -> tuple[BoardBatch, list[bbc.BoardCore]]:
    """Build mid-game boards with a synced list of BoardCore references."""
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    board = BoardBatch.empty(B, DEVICE)
    cores = [bbc.BoardCore() for _ in range(B)]
    for _ in range(n_rounds):
        actions = torch.randint(0, 7, (B,), device=DEVICE, generator=g)
        board.play_columns(actions)
        for i in range(B):
            cores[i].play(int(actions[i].item()))
        done = board.done()
        board.reset(done)
        for i in range(B):
            if cores[i].hasWin() or cores[i].movesLeft() <= 0:
                cores[i].setRawState(0, 0, 42)
    return board, cores


# ---------------------------------------------------------------------------
# 1) NTupleNetwork: output range
# ---------------------------------------------------------------------------
def test_ntuple_network_output_in_tanh_range():
    """forward() must return values in [-1, 1] (tanh activation)."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(256)

    with torch.no_grad():
        vals = net(board)

    assert vals.shape == (256,)
    assert vals.min() >= -1.0 and vals.max() <= 1.0, (
        f"Values outside [-1, 1]: min={vals.min():.6f}, max={vals.max():.6f}"
    )
    print("PASSED: test_ntuple_network_output_in_tanh_range")


# ---------------------------------------------------------------------------
# 2) NTupleNetwork: zero weights -> zero output
# ---------------------------------------------------------------------------
def test_ntuple_network_zero_weights_give_zero():
    """With W=0 the network must output exactly 0 for every board."""
    net = _mk_net(random_weights=False)  # W is all zeros
    board = _build_midgame_boards(128)

    with torch.no_grad():
        vals = net(board)

    assert (vals == 0.0).all(), (
        f"Expected all zeros, got min={vals.min()}, max={vals.max()}"
    )
    print("PASSED: test_ntuple_network_zero_weights_give_zero")


# ---------------------------------------------------------------------------
# 3) NTupleNetwork: mirror symmetry (value of board == value of mirror)
# ---------------------------------------------------------------------------
def test_ntuple_network_mirror_symmetry():
    """The network uses W + W_mirror, so v(board) == v(mirror(board))."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(256)
    mir = board.mirror()

    with torch.no_grad():
        v_orig = net(board)
        v_mir = net(mir)

    assert torch.allclose(v_orig, v_mir, atol=1e-6), (
        f"Mirror asymmetry: max diff = {(v_orig - v_mir).abs().max():.8f}"
    )
    print("PASSED: test_ntuple_network_mirror_symmetry")


# ---------------------------------------------------------------------------
# 4) NTupleNetwork: save / load roundtrip
# ---------------------------------------------------------------------------
def test_ntuple_network_save_load_roundtrip():
    """Save + load must produce bit-identical forward pass results."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(64)

    with torch.no_grad():
        v_before = net(board).clone()

    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        path = f.name
    try:
        net.save(path)
        net2 = NTupleNetwork.load(path, device="cpu")
        net2.eval()
        with torch.no_grad():
            v_after = net2(board)
        assert torch.equal(v_before, v_after), "Forward pass differs after save/load"
    finally:
        os.remove(path)
    print("PASSED: test_ntuple_network_save_load_roundtrip")


# ---------------------------------------------------------------------------
# 5) best_afterstate_values: returned moves are legal one-hot masks
# ---------------------------------------------------------------------------
def test_bav_returns_legal_onehot_moves():
    """Every non-zero move from best_afterstate_values must be a legal one-hot landing square."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(512)

    with torch.no_grad():
        best_mv, best_val = best_afterstate_values(board, net, use_non_losing=False)

    legal = board.generate_legal_moves()
    active = best_mv != 0

    # Subset of legal moves
    assert ((best_mv[active] & legal[active]) == best_mv[active]).all(), (
        "Some returned moves are not legal landing squares"
    )
    # One-hot (single bit set) or zero
    one_hot_or_zero = (best_mv == 0) | ((best_mv & (best_mv - 1)) == 0)
    assert one_hot_or_zero.all(), "Some returned moves have multiple bits set"
    print("PASSED: test_bav_returns_legal_onehot_moves")


# ---------------------------------------------------------------------------
# 6) best_afterstate_values: moves are within non-losing set
# ---------------------------------------------------------------------------
def test_bav_moves_subset_of_non_losing():
    """With use_non_losing=True, returned moves must be within non-losing moves."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(512)

    nl_moves = board.generate_non_losing_moves()

    with torch.no_grad():
        best_mv, _ = best_afterstate_values(board, net, use_non_losing=True)

    active = best_mv != 0
    assert ((best_mv[active] & nl_moves[active]) == best_mv[active]).all(), (
        "Some returned moves are outside the non-losing move set"
    )
    print("PASSED: test_bav_moves_subset_of_non_losing")


# ---------------------------------------------------------------------------
# 7) best_afterstate_values: randomize=True still returns legal moves
# ---------------------------------------------------------------------------
def test_bav_random_mode_still_legal():
    """With randomize=all True, returned moves must still be legal and one-hot."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(512)

    randomize = torch.ones(512, dtype=torch.bool)
    with torch.no_grad():
        rand_mv, rand_val = best_afterstate_values(
            board,
            net,
            randomize=randomize,
            use_non_losing=False,
        )

    legal = board.generate_legal_moves()
    active = rand_mv != 0
    assert ((rand_mv[active] & legal[active]) == rand_mv[active]).all(), (
        "Random moves are not all legal"
    )
    one_hot_or_zero = (rand_mv == 0) | ((rand_mv & (rand_mv - 1)) == 0)
    assert one_hot_or_zero.all(), "Random moves are not one-hot"
    print("PASSED: test_bav_random_mode_still_legal")


# ---------------------------------------------------------------------------
# 8) best_afterstate_values: terminal afterstates have correct reward as value
# ---------------------------------------------------------------------------
def test_bav_terminal_rewards():
    """For terminal afterstates, V_new must be the exact game reward (+1/-1/0)."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(2048, n_rounds=20)

    with torch.no_grad():
        best_mv, best_val = best_afterstate_values(board, net, use_non_losing=False)

    # Apply the chosen moves to copies
    tmp = BoardBatch(
        all_tokens=board.all_tokens.clone(),
        active_tokens=board.active_tokens.clone(),
        moves_left=board.moves_left.clone(),
    )
    tmp.play_masks(best_mv)
    done = tmp.done()
    reward = tmp.reward()

    n_terminal = done.sum().item()
    assert n_terminal > 0, "No terminal afterstates found; increase n_rounds or B"

    # For terminal states, best_val must match the reward
    terminal_vals = best_val[done]
    terminal_rewards = reward[done]
    assert torch.equal(terminal_vals, terminal_rewards), (
        f"Terminal values don't match rewards.\n"
        f"  vals    = {terminal_vals[:10].tolist()}\n"
        f"  rewards = {terminal_rewards[:10].tolist()}"
    )

    # Verify specific reward values
    has_win = tmp.has_win()
    yellow_won = done & has_win & ((tmp.moves_left.to(torch.int64) & 1) == 1)
    red_won = done & has_win & ((tmp.moves_left.to(torch.int64) & 1) == 0)
    draw = done & (~has_win) & (tmp.moves_left == 0)

    if yellow_won.any():
        assert (best_val[yellow_won] == 1.0).all(), "Yellow wins should have value +1.0"
    if red_won.any():
        assert (best_val[red_won] == -1.0).all(), "Red wins should have value -1.0"
    if draw.any():
        assert (best_val[draw] == 0.0).all(), "Draws should have value 0.0"

    print(
        f"  Verified {n_terminal} terminal states: "
        f"{yellow_won.sum()} yellow wins, {red_won.sum()} red wins, {draw.sum()} draws"
    )
    print("PASSED: test_bav_terminal_rewards")


# ---------------------------------------------------------------------------
# 9) best_afterstate_values: cross-check with BoardCore (the original disabled code)
# ---------------------------------------------------------------------------
def test_bav_consistency_with_core():
    """Play best_afterstate_values moves on BoardCore in parallel and verify state match.

    This is the proper test version of the `if False:` verification block.
    """
    net = _mk_net(random_weights=True)
    B = 200
    board, cores = _build_synced_boards(B, n_rounds=15)

    with torch.no_grad():
        best_mv, V_new = best_afterstate_values(
            board,
            net,
            use_non_losing=False,
            no_grad=True,
        )

    for b_idx in range(B):
        mv = int(best_mv[b_idx].item())
        if mv == 0:
            continue  # board already terminal or forced-loss (no moves)

        col = move_mask_to_column(mv)
        assert cores[b_idx].isLegalMove(col), (
            f"Board {b_idx}: move column {col} is not legal in core"
        )

        # Play on core
        ok = cores[b_idx].play(col)
        assert ok, f"Board {b_idx}: core.play({col}) returned False"
        bb_done = cores[b_idx].hasWin() or cores[b_idx].movesLeft() <= 0

        # Play on BoardBatch copy to check state equivalence
        tmp = BoardBatch(
            all_tokens=board.all_tokens[b_idx : b_idx + 1].clone(),
            active_tokens=board.active_tokens[b_idx : b_idx + 1].clone(),
            moves_left=board.moves_left[b_idx : b_idx + 1].clone(),
        )
        tmp.play_masks(torch.tensor([mv], dtype=torch.int64))

        # State equivalence
        a_core, b_core, m_core = cores[b_idx].rawState()
        assert int(a_core) == int(tmp.all_tokens[0].item()), (
            f"Board {b_idx}: all_tokens mismatch"
        )
        assert int(b_core) == int(tmp.active_tokens[0].item()), (
            f"Board {b_idx}: active_tokens mismatch"
        )
        assert int(m_core) == int(tmp.moves_left[0].item()), (
            f"Board {b_idx}: moves_left mismatch"
        )

        tmp_done = bool(tmp.done()[0].item())
        assert bb_done == tmp_done, (
            f"Board {b_idx}: done mismatch (core={bb_done}, batch={tmp_done})"
        )

        # Terminal reward check (same logic as the original disabled code)
        if bb_done:
            v = float(V_new[b_idx].item())
            if cores[b_idx].hasWin() and cores[b_idx].movesLeft() % 2 == 0:
                assert v == -1.0, f"Board {b_idx}: red win should be -1, got {v}"
            elif cores[b_idx].hasWin() and cores[b_idx].movesLeft() % 2 == 1:
                assert v == 1.0, f"Board {b_idx}: yellow win should be +1, got {v}"
            elif cores[b_idx].movesLeft() <= 0:
                assert v == 0.0, f"Board {b_idx}: draw should be 0, got {v}"

        # Reset core if done (to match the original verification pattern)
        if bb_done:
            cores[b_idx].setRawState(0, 0, 42)

    print("PASSED: test_bav_consistency_with_core")


# ---------------------------------------------------------------------------
# 10) best_afterstate_values: greedy choice is optimal (accounting for tiebreak noise)
# ---------------------------------------------------------------------------
def test_bav_greedy_is_optimal():
    """The greedy move's un-noised value must be within noise tolerance of the true optimum.

    best_afterstate_values adds tiebreak noise eps=1e-4 * randn, so we allow ~5e-4 tolerance.
    """
    net = _mk_net(random_weights=True)
    B = 128
    board = _build_midgame_boards(B, n_rounds=12)

    with torch.no_grad():
        best_mv, best_val = best_afterstate_values(board, net, use_non_losing=False)

    yellow_to_move = (board.moves_left.to(torch.int64) & 1) == 0

    # Compute the true best value (without noise) by enumerating all legal moves
    legal_mask = board.generate_legal_moves()
    neg_inf = torch.full((B,), float("-inf"))
    pos_inf = torch.full((B,), float("inf"))
    true_best = torch.where(yellow_to_move, neg_inf, pos_inf).to(torch.float32)

    for mv in board.iter_move_masks(legal_mask):
        active = mv != 0
        if not active.any():
            break
        tmp = BoardBatch(
            all_tokens=board.all_tokens.clone(),
            active_tokens=board.active_tokens.clone(),
            moves_left=board.moves_left.clone(),
        )
        tmp.play_masks(mv)
        r = tmp.reward().to(torch.float32)
        with torch.no_grad():
            v = net(tmp).to(torch.float32)
        val = torch.where(torch.isnan(r), v, r)

        better = torch.where(yellow_to_move, val > true_best, val < true_best) & active
        true_best = torch.where(better, val, true_best)

    # The chosen value (noise-free recomputed) should be close to true_best
    # Recompute the chosen move's noise-free value
    tmp_chosen = BoardBatch(
        all_tokens=board.all_tokens.clone(),
        active_tokens=board.active_tokens.clone(),
        moves_left=board.moves_left.clone(),
    )
    tmp_chosen.play_masks(best_mv)
    r_chosen = tmp_chosen.reward().to(torch.float32)
    with torch.no_grad():
        v_chosen = net(tmp_chosen).to(torch.float32)
    chosen_noisefree = torch.where(torch.isnan(r_chosen), v_chosen, r_chosen)

    has_move = best_mv != 0
    tol = 5e-4  # noise is ~1e-4 * randn; 5-sigma margin

    yellow_ok = ~has_move | ~yellow_to_move | (chosen_noisefree >= true_best - tol)
    red_ok = ~has_move | yellow_to_move | (chosen_noisefree <= true_best + tol)

    assert yellow_ok.all(), (
        f"Yellow suboptimality beyond tolerance: "
        f"worst gap = {(true_best[yellow_to_move & has_move] - chosen_noisefree[yellow_to_move & has_move]).max():.6f}"
    )
    assert red_ok.all(), (
        f"Red suboptimality beyond tolerance: "
        f"worst gap = {(chosen_noisefree[~yellow_to_move & has_move] - true_best[~yellow_to_move & has_move]).max():.6f}"
    )

    print(f"  Checked {has_move.sum()} boards with moves")
    print("PASSED: test_bav_greedy_is_optimal")


# ---------------------------------------------------------------------------
# 11) Training-loop invariants: moves are always legal, done boards have rewards
# ---------------------------------------------------------------------------
def test_training_loop_invariants():
    """Simulate a few training steps and verify the two `if False:` guards from the loop:
    1) All returned moves are legal (play_masks returns True)
    2) Done boards always have a non-NaN reward
    """
    net = _mk_net(random_weights=True)
    B = 1024
    board = BoardBatch.empty(B, DEVICE)
    epsilon = 0.1

    for step in range(50):
        randomize = torch.rand((B,), device=DEVICE) < epsilon

        with torch.no_grad():
            best_mv, V_new = best_afterstate_values(
                board,
                net,
                randomize=randomize,
                use_non_losing=False,
                no_grad=True,
            )

        legal = board.play_masks(best_mv)

        # Invariant 1: all moves should be accepted as legal
        assert legal.all(), f"Step {step}: {(~legal).sum()} illegal moves returned"

        # Invariant 2: done boards must have non-NaN reward
        done = board.done()
        reward = board.reward()
        done_nan = done & torch.isnan(reward)
        assert not done_nan.any(), (
            f"Step {step}: {done_nan.sum()} done boards have NaN reward"
        )

        board.reset(done)

    print("PASSED: test_training_loop_invariants")


# ---------------------------------------------------------------------------
# 12) NTupleNetwork: gradient flows through all components
# ---------------------------------------------------------------------------
def test_ntuple_network_gradient_flow():
    """A single forward + backward step must produce non-zero gradients on W."""
    net = NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST)
    # Small random weights to avoid zero-gradient at tanh saturation
    with torch.no_grad():
        net.W.normal_(0, 0.01)
    net.train()

    board = _build_midgame_boards(64)
    vals = net(board)
    loss = vals.sum()
    loss.backward()

    assert net.W.grad is not None, "W.grad is None"
    assert (net.W.grad != 0).any(), "W.grad is all zeros (no gradient flow)"
    print(f"  Non-zero grad entries: {(net.W.grad != 0).sum()} / {net.W.grad.numel()}")
    print("PASSED: test_ntuple_network_gradient_flow")


# ===========================================================================
# Run all tests
# ===========================================================================
def run_all_tests():
    tests = [
        test_ntuple_network_output_in_tanh_range,
        test_ntuple_network_zero_weights_give_zero,
        test_ntuple_network_mirror_symmetry,
        test_ntuple_network_save_load_roundtrip,
        test_ntuple_network_gradient_flow,
        test_bav_returns_legal_onehot_moves,
        test_bav_moves_subset_of_non_losing,
        test_bav_random_mode_still_legal,
        test_bav_terminal_rewards,
        test_bav_consistency_with_core,
        test_bav_greedy_is_optimal,
        test_training_loop_invariants,
    ]

    passed, failed = 0, 0
    for t in tests:
        try:
            t()
            passed += 1
        except Exception as e:
            print(f"FAILED: {t.__name__}: {e}")
            import traceback

            traceback.print_exc()
            failed += 1

    print(f"\n{'=' * 60}")
    print(f"Results: {passed} passed, {failed} failed, {passed + failed} total")
    if failed == 0:
        print("All tests passed!")


run_all_tests()